# 01 · Data preparation and SQL layer
**Scope reminder:** PD model on public Lending Club data. Not a Basel/IRB framework; LGD/EAD are illustrative proxies.

**What this notebook does:** loads the raw CSV into DuckDB, builds the typed `model_frame` (`sql/03_model_frame.sql`), applies the status mapping, the 36-month filter and the maturity flag, and documents the missing-value rules.

### Why these choices (see `docs/decision_log.md`)
- **DuckDB, not SQLite/pandas read_csv:** ~2M rows x ~150 columns. DuckDB reads only the columns we select and everything lands as VARCHAR first, so mixed-type columns cannot silently corrupt the load; typing is explicit in SQL.
- **36-month loans only (D-01):** the file is one 2018Q4 snapshot (outcomes, not a panel). 36m loans issued up to Dec-2015 are fully matured; 60m loans only up to Dec-2013. Mixing terms mixes maturity windows and makes default rates incomparable across vintages.
- **Target (D-02):** Charged Off/Default = 1, Fully Paid = 0. Current/Late/In Grace have **unknown** final outcome, so they are excluded from modelling rather than assumed good or bad. They are kept in `model_frame` (target NULL) because PSI needs features, not outcomes.

In [ ]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src.utils.config import load_config, set_seeds
cfg = load_config(); set_seeds(cfg["seed"])
pd.set_option("display.float_format", "{:.4f}".format); pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)
print("config seed:", cfg["seed"], "| raw file:", cfg["paths"]["raw_csv"])

In [ ]:
from src.data.loader import build_database, load_model_frame, sql_query
REBUILD = False          # set True after changing anything in sql/ or the raw file
build_database(cfg, rebuild=REBUILD)
sql_query(cfg, "SELECT trim(term) AS term, count(*) AS loans FROM raw_loans GROUP BY 1 ORDER BY 2 DESC")

Status mapping: every status and what happens to it.

**Load audit.** `ignore_errors=true` skips malformed CSV rows silently (e.g. the free-text `desc` column). `build_database` therefore asserts a raw-row floor (`loader.min_raw_rows`), writes `reports/load_audit.json`, and deletes a failed build so it cannot be reused. Check it before trusting anything downstream.

In [ ]:
from src.data.loader import load_audit
a = load_audit(cfg); print({k: v for k, v in a.items() if k != "raw_rows_by_loan_status"}); pd.Series(a.get("raw_rows_by_loan_status", {}), name="raw_rows").to_frame()

In [ ]:
sql_query(cfg, """SELECT loan_status, count(*) AS loans, any_value(target) AS target, any_value(is_resolved) AS is_resolved
                  FROM model_frame GROUP BY 1 ORDER BY 2 DESC""")

Vintage, resolution share and maturity flag, straight from SQL.

In [ ]:
sql_query(cfg, """SELECT vintage_year, count(*) AS loans, avg(is_resolved::INT) AS resolved_share,
                         avg(fully_matured::INT) AS fully_matured_share, avg(target) AS default_rate_resolved
                  FROM model_frame GROUP BY 1 ORDER BY 1""")

### Documented missing-value and cleaning rules (`src/data/cleaning.py`)
| Rule | Treatment | Justification |
|---|---|---|
| R1 `emp_length` | ordered buckets; missing -> `Unknown` | missingness is itself informative; avoids dropping rows |
| R2 numeric NaN | **kept**, own WoE bin | imputation would hide the missingness signal; WoE handles it natively |
| R3 impossible values | `dti<0 or >100`, `annual_inc<=0`, `revol_util<0`, `loan_amnt<=0` -> NaN | 999 `dti` is a known sentinel |
| R4 winsorising | train 0.5/99.5 percentiles, re-applied (never refit) | stops tails driving bin edges; no leakage from validation |
| R5 rare categories | pooled to `_POOLED_` inside the binner (<5% of train) | stable WoE |

In [ ]:
from src.data.cleaning import basic_clean, missingness_report
raw = load_model_frame(cfg)
print("rows:", len(raw))
imp = pd.Series({"dti<0 or >100": ((raw.dti < 0) | (raw.dti > 100)).sum(), "annual_inc<=0": (raw.annual_inc <= 0).sum(),
                 "revol_util<0": (raw.revol_util < 0).sum(), "loan_amnt<=0": (raw.loan_amnt <= 0).sum()})
print("impossible values found (set to NaN by R3):"); print(imp.to_string())
df = basic_clean(raw)
cols = cfg["features"]["numeric"] + ["emp_length_raw"]
missingness_report(df, cols).loc[2007:]